In [34]:
import pandas as pd
import plotly.express as px
import numpy as np
import matplotlib.pyplot as plt
import plotly.io as pio

In [35]:
datapath = "data/"

df_history = pd.read_csv(f"{datapath}europe_yearly_full_release_long_format.csv")
df_prices = pd.read_csv(f"{datapath}european_wholesale_electricity_price_data_monthly.csv")

xls = pd.ExcelFile(f"{datapath}targets_download.xlsx")
print(f"found {len(xls.sheet_names)} sheets in targets file: {xls.sheet_names}")

found 4 sheets in targets file: ['raw_data_long', 'sources', 'capacity_target_wide', 'share_target_wide']


In [36]:
df_targets = pd.read_excel(f"{datapath}targets_download.xlsx", sheet_name='raw_data_long')

In [37]:
df_history.head()

,Area,ISO 3 code,Year,Area type,Continent,Ember region,EU,OECD,G20,G7,ASEAN,Category,Subcategory,Variable,Unit,Value,YoY absolute change,YoY % change
0,Austria,AUT,1990,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity demand,Demand,Demand,TWh,48.81,NaN,NaN
1,Austria,AUT,1990,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity demand,Demand per capita,Demand per capita,MWh,6.36,NaN,NaN
2,Austria,AUT,1990,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Aggregate fuel,Clean,%,66.25,NaN,NaN
3,Austria,AUT,1990,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Aggregate fuel,Coal,%,12.56,NaN,NaN
4,Austria,AUT,1990,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Aggregate fuel,Fossil,%,33.75,NaN,NaN


In [38]:
datasets = [df_history, df_targets, df_prices]

for df in datasets:
    print(f"Shape of df: {df.shape}")
    print(f"Available columns in df: {df.columns}")

Shape of df: (69310, 18)
Available columns in df: Index(['Area', 'ISO 3 code', 'Year', 'Area type', 'Continent', 'Ember region',
       'EU', 'OECD', 'G20', 'G7', 'ASEAN', 'Category', 'Subcategory',
       'Variable', 'Unit', 'Value', 'YoY absolute change', 'YoY % change'],
      dtype='str')
Shape of df: (2143, 8)
Available columns in df: Index(['COUNTRY_CODE', 'TARGET_YEAR', 'FUEL_CATEGORY', 'METRIC', 'VALUE',
       'SOURCE_ID', 'NUMBER_OF_UPDATES', 'COLLECTION_YEAR'],
      dtype='str')
Shape of df: (3801, 4)
Available columns in df: Index(['Country', 'ISO3 Code', 'Date', 'Price (EUR/MWhe)'], dtype='str')


In [39]:
df_history_cleaned = df_history[
    (df_history['Year'] == 2023) & 
    (df_history['Category'] == 'Electricity generation') &
    (df_history['Unit'] == 'TWh')
].copy()

df_history_cleaned[df_history_cleaned["Area"] == "Austria"]

,Area,ISO 3 code,Year,Area type,Continent,Ember region,EU,OECD,G20,G7,ASEAN,Category,Subcategory,Variable,Unit,Value,YoY absolute change,YoY % change
1758,Austria,AUT,2023,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Aggregate fuel,Clean,TWh,60.46,10.04,19.91
1759,Austria,AUT,2023,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Aggregate fuel,Coal,TWh,0.00,0.00,NaN
1760,Austria,AUT,2023,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Aggregate fuel,Fossil,TWh,10.82,-3.25,-23.10
1761,Austria,AUT,2023,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Aggregate fuel,"Hydro, bioenergy and other renewables",TWh,45.23,5.85,14.86
1762,Austria,AUT,2023,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Aggregate fuel,Renewables,TWh,60.46,10.04,19.91
1763,Austria,AUT,2023,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Aggregate fuel,Wind,TWh,8.34,1.09,15.03
1764,Austria,AUT,2023,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Aggregate fuel,Wind and solar,TWh,15.23,4.19,37.95
1775,Austria,AUT,2023,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Fuel,Bioenergy,TWh,4.56,-0.19,-4.00
1776,Austria,AUT,2023,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Fuel,Gas,TWh,7.50,-3.35,-30.88
1777,Austria,AUT,2023,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Fuel,Hard coal,TWh,0.00,0.00,NaN


In [40]:
df_targets.columns

Index(['COUNTRY_CODE', 'TARGET_YEAR', 'FUEL_CATEGORY', 'METRIC', 'VALUE',
       'SOURCE_ID', 'NUMBER_OF_UPDATES', 'COLLECTION_YEAR'],
      dtype='str')

In [41]:
# identify available target metrics and find the one that matches the history data
print(f"Metrics available in df_targets: {df_targets["METRIC"].unique()}")

# gonna use 'generation_twh'

df_targets_cleaned = df_targets[
    (df_targets["METRIC"] == "generation_twh") &
    (df_targets["TARGET_YEAR"] == 2030)
].copy()

Metrics available in df_targets: <StringArray>
[       'capacity_total_gw', 'share_of_consumption_pct',
           'generation_twh',  'share_of_generation_pct',
   'capacity_additional_mw',               'demand_twh',
               'import_twh']
Length: 7, dtype: str


In [42]:
# pick columns and rename to make the data comparable

df_history_cleaned = df_history_cleaned[["ISO 3 code", "Variable", "Value"]]
df_targets_cleaned = df_targets_cleaned[["COUNTRY_CODE", "FUEL_CATEGORY", "VALUE"]]

df_history_cleaned = df_history_cleaned.rename(columns = {"ISO 3 code": "iso", "Variable": "source", "Value": "value"})
df_targets_cleaned = df_targets_cleaned.rename(columns = {"COUNTRY_CODE": "iso", "FUEL_CATEGORY": "source", "VALUE": "value"})

In [43]:
df_history_cleaned

,iso,source,value
1758,AUT,Clean,60.46
1759,AUT,Coal,0.00
1760,AUT,Fossil,10.82
1761,AUT,"Hydro, bioenergy and other renewables",45.23
1762,AUT,Renewables,60.46
...,...,...,...
69187,GBR,Offshore wind,49.65
69188,GBR,Onshore wind,32.50
69189,GBR,Other fossil,10.82
69190,GBR,Solar,13.88


In [44]:
df_history_cleaned["source"].unique()

<StringArray>
[                                'Clean',
                                  'Coal',
                                'Fossil',
 'Hydro, bioenergy and other renewables',
                            'Renewables',
                                  'Wind',
                        'Wind and solar',
                             'Bioenergy',
                                   'Gas',
                             'Hard coal',
                                 'Hydro',
                               'Lignite',
                               'Nuclear',
                          'Onshore wind',
                          'Other fossil',
                      'Other renewables',
                                 'Solar',
                      'Total generation',
                         'Offshore wind']
Length: 19, dtype: str

In [45]:
df_targets_cleaned["source"].unique()

<StringArray>
[                          'Solar',                            'Wind',
                       'Bioenergy',                           'Hydro',
                'Other Renewables',                            'Coal',
                             'Gas',                         'Nuclear',
                    'Other Fossil',                      'Renewables',
                    'Onshore Wind',                   'Offshore Wind',
                          'Fossil', 'Hydro, bio and other renewables',
                          'Demand',            'Non-hydro renewables',
              'Rest of renewables',                         'Storage',
                    'Pumped hydro',                         'Battery',
                         'Imports']
Length: 21, dtype: str

In [46]:
# sort the "source" column in fossil and renewables
# note on columns not considered:
# battery/storage/pumped hydro: no info about the source, 

RENEWABLES = {
    "solar", "wind", "onshore wind", "offshore wind", "wind and solar",
    "bioenergy", "hydro", "other renewables", "renewables",
    "hydro, bio and other renewables", "hydro, bioenergy and other renewables",
    "non-hydro renewables", "rest of renewables",
}

RENEWABLES_EVALUATED = {
    "solar",
}

FOSSIL = {
    "coal", "hard coal", "lignite", "gas", "other fossil", "fossil",
}

def categorize(value):
    if pd.isna(value):
        return pd.NA
    v = str(value).strip().lower()
    if v in RENEWABLES_EVALUATED:
        return "Renewables"
    if v in FOSSIL:
        return "Fossil"
    return "Other"

df_targets_cleaned["category"] = df_targets_cleaned["source"].map(categorize)
df_history_cleaned["category"] = df_history_cleaned["source"].map(categorize)

print(f"Unique values in df_targets_cleaned: {df_targets_cleaned["category"].unique()}")
print(f"Unique values in df_history_cleaned: {df_history_cleaned["category"].unique()}")

df_targets_cleaned = df_targets_cleaned.groupby(by=["category", "iso"]).sum()

df_history_cleaned = df_history_cleaned.groupby(by=["category", "iso"]).sum()

Unique values in df_targets_cleaned: <StringArray>
['Renewables', 'Other', 'Fossil']
Length: 3, dtype: str
Unique values in df_history_cleaned: <StringArray>
['Other', 'Fossil', 'Renewables']
Length: 3, dtype: str


In [47]:
df_gap = pd.merge(df_history_cleaned, df_targets_cleaned, on=["iso", "category"], how="inner", suffixes=["_history", "_target"])

df_gap["gap_twh"] = df_gap["value_target"] - df_gap["value_history"]
df_gap["pct_missing"] = (df_gap["gap_twh"] / df_gap["value_target"]) * 100

In [48]:
df_gap

,,source_history,value_history,source_target,value_target,gap_twh,pct_missing
iso,category,,,,,,
AUT,Fossil,CoalFossilGasHard coalLigniteOther fossil,21.64,GasFossilGasFossilGas,19.000,-2.640,-13.894737
BEL,Fossil,CoalFossilGasHard coalLigniteOther fossil,43.48,Gas,24.650,-18.830,-76.389452
BGR,Fossil,CoalFossilGasHard coalLigniteOther fossil,39.47,FossilFossil,10.474,-28.996,-276.837884
CHE,Fossil,FossilOther fossil,2.08,Other Fossil,1.600,-0.480,-30.000000
CYP,Fossil,CoalFossilGasHard coalLigniteOther fossil,8.48,CoalGasCoalGas,7.564,-0.916,-12.109995
...,...,...,...,...,...,...,...
ROU,Renewables,Solar,2.23,SolarSolar,21.700,19.470,89.723502
SVK,Renewables,Solar,0.61,Solar,1.470,0.860,58.503401
SVN,Renewables,Solar,0.98,SolarSolar,3.757,2.777,73.915358


In [50]:
# Create an Interactive Bar Chart
df_gap = df_gap.reset_index()
df_graph =  df_gap[df_gap["category"] == 'Renewables'].sort_values(by='value_target', ascending=True)
df_graph = df_graph.rename(columns={"iso": "country", "value_history": "current_twh"})

fig = px.bar(
    df_graph, 
    y='country', 
    x=['current_twh', 'gap_twh'], 
    title='<b>The Solar Reality Gap (2023 vs 2030 Target)</b><br><i>Dark Blue = Current Generation, Light Blue = Remaining Gap to Target</i>',
    labels={'value': 'Generation (TWh)', 'country': 'Country', 'variable': 'Metric'},
    orientation='h',  # Horizontal bars are easier to read for country names
    height=800,       # Tall chart to fit all countries
    color_discrete_map={'current_twh': '#1f77b4', 'gap_twh': '#aec7e8'} # Dark vs Light Blue
)

# Customizing the Layout for a Professional Look
fig.update_layout(
    xaxis_title="Electricity Generation (TWh)",
    legend_title_text="",
    showlegend=True,
    template="plotly_white",
    hovermode="y unified" # Shows both values when hovering over a country
)

# Fix the legend names
new_names = {'current_twh': 'Current (2023)', 'gap_twh': 'Remaining Gap'}
fig.for_each_trace(lambda t: t.update(name = new_names[t.name]))

fig.show()

## Prices

In [51]:
df_prices

,Country,ISO3 Code,Date,Price (EUR/MWhe)
0,Austria,AUT,2015-01-01,29.94
1,Belgium,BEL,2015-01-01,42.33
2,Czechia,CZE,2015-01-01,29.47
3,Denmark,DNK,2015-01-01,27.12
4,Estonia,EST,2015-01-01,33.84
...,...,...,...,...
3796,Slovakia,SVK,2026-02-01,122.36
3797,Slovenia,SVN,2026-02-01,124.98
3798,Spain,ESP,2026-02-01,20.37
3799,Sweden,SWE,2026-02-01,102.59
